# Parte 1. Scraping de decretos de emergencia
**Grupo 6 - 1 de enero al 31 de mayo de 2023**

## Paso 1. Importar y descargar `robots.txt`
Un pedido HTTP devuelve una respuesta. `raise_for_status()` detiene el proceso si no fue
exitoso. La pausa limita la frecuencia de consulta; no sustituye a `WebDriverWait`.

In [1]:
import pandas as pd
import requests
import json, re, time, calendar
from datetime import datetime, timezone
from urllib.parse import urlencode
from bs4 import BeautifulSoup
from IPython.display import display
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from pathlib import Path
BASE = Path.cwd()
if (BASE / "assignment_2").is_dir():
    BASE = BASE / "assignment_2"
DATOS = BASE / "datos"
FUENTES = DATOS / "fuentes"
FUENTES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 60)

departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
"Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco", "Ica", "Junín",
"La Libertad", "Lambayeque", "Lima", "Loreto", "Madre de Dios", "Moquegua",
"Pasco", "Piura", "Puno", "San Martín", "Tacna", "Tumbes", "Ucayali"]

def guardar_json(ruta, contenido):
    ruta.write_text(json.dumps(contenido, ensure_ascii=False, indent=2), encoding="utf-8")

def descargar_texto(url, ruta, parametros=None):
    # Reutilizar la respuesta original evita repetir consultas al ejecutar otra vez.
    if ruta.exists():
        return ruta.read_text(encoding="utf-8")
    respuesta = requests.get(url, params=parametros,
                             headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    respuesta.raise_for_status()
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(respuesta.text, encoding="utf-8")
    guardar_json(ruta.with_suffix(ruta.suffix + ".meta.json"), {
        "url": respuesta.url,
        "consultado_utc": datetime.now(timezone.utc).isoformat(),
        "status_code": respuesta.status_code,
    })
    time.sleep(1.1)  # Al menos un segundo entre pedidos.
    return respuesta.text

In [2]:
robots = descargar_texto("https://www.gob.pe/robots.txt", FUENTES / "robots.txt")
print(robots)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



Para `User-agent: *`, se prohíben `/admin/`
y las URL que incluyen los parámetros `?sheet=` o `&sheet=`. Para `AhrefsBot` se prohíbe
todo el sitio (`Disallow: /`). Los grupos de `GPTBot` y `OAI-SearchBot` repiten las
restricciones de administración y hojas, y piden `Crawl-delay: 5` y `Crawl-delay: 2`,
respectivamente. Esas esperas se refieren a los agentes nombrados.

`/busquedas` no figura entre las rutas prohibidas para el agente general, y nuestra URL
no incluye `sheet`. Aun así, se esperan 2.1 segundos entre páginas del buscador y 1.1
segundos entre páginas de normas: cada consulta consume recursos del servidor.
La regla se revisó en la copia actual; no se supone que `robots.txt` sea inmutable.

## Paso 2. Leer una tarjeta antes de generalizar
Cada resultado es un `article`. La búsqueda del enlace, fecha y descripción se hace
dentro de ese artículo, para no mezclar datos de normas distintas. El enlace principal
está en `h4 a`; la fecha se reconoce porque empieza con `Publicado:`.

In [3]:
def leer_articulo(articulo):
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = None
    for elemento in articulo.find_elements(By.TAG_NAME, "span"):
        if elemento.text.startswith("Publicado:"):
            fecha = elemento.text.replace("Publicado:", "").strip()
            break
    assert fecha is not None, "No se encontró la fecha dentro del artículo."
    return {
        "numero": enlace.text.strip(),
        "fecha_texto": fecha,
        "titulo_lista": articulo.find_element(
            By.CSS_SELECTOR, 'p[id$="-description"]').text.strip(),
        "enlace": enlace.get_attribute("href"),
    }

## Paso 3. Extraer un mes completo y comprobar la paginación
El total se lee del texto visible "N Resultados". Si faltan filas, se busca el enlace
a la siguiente página. Si no aparece, el programa se detiene: no presenta una extracción
incompleta como exitosa. En esta captura, cada mes entró en una sola página.

In [4]:
def extraer_mes(navegador, mes):
    parametros = {
        "contenido[]": "normas", "institucion[]": "pcm",
        "term": "estado de emergencia precipitaciones", "sort_by": "recent",
        "desde": f"01-{mes:02d}-2023",
        "hasta": f"{calendar.monthrange(2023, mes)[1]:02d}-{mes:02d}-2023",
    }
    url = "https://www.gob.pe/busquedas?" + urlencode(parametros)
    filas, paginas, firmas = [], [], set()
    total = None
    while url:
        time.sleep(2.1)
        navegador.get(url)
        espera = WebDriverWait(navegador, 45)
        patron = r"([\d.,]+)\s+Resultados\b"
        espera.until(lambda n: re.search(
            patron, n.find_element(By.TAG_NAME, "body").text, re.I))
        texto = navegador.find_element(By.TAG_NAME, "body").text
        total_pagina = int(re.sub(r"\D", "", re.search(patron, texto, re.I).group(1)))
        if total is None:
            total = total_pagina
        assert total == total_pagina, "Cambió el total entre páginas."
        if total > 0:
            espera.until(EC.presence_of_all_elements_located((By.TAG_NAME, "article")))
        nuevas = [leer_articulo(a) for a in navegador.find_elements(By.TAG_NAME, "article")]
        firma = tuple(f["enlace"] for f in nuevas)
        assert firma not in firmas, "Se repitió la misma página."
        firmas.add(firma)
        filas.extend(nuevas)
        pagina = len(paginas) + 1
        (FUENTES / f"busqueda_2023_{mes:02d}_p{pagina}.html").write_text(
            navegador.page_source, encoding="utf-8")
        paginas.append({"url": navegador.current_url, "extraidos": len(nuevas)})
        if len(filas) >= total:
            break
        candidatos = []
        for a in navegador.find_elements(By.CSS_SELECTOR, "a[href]"):
            href = a.get_attribute("href") or ""
            if re.search(r"[?&]page=" + str(pagina + 1) + r"(?:&|$)", href):
                candidatos.append(href)
        assert candidatos, "Faltan resultados y no se encontró la página siguiente."
        url = candidatos[0]
    assert len(filas) == total
    assert len({f["enlace"] for f in filas}) == total
    captura = {
        "mes": f"2023-{mes:02d}", "resultados_totales": total,
        "consultado_utc": datetime.now(timezone.utc).isoformat(),
        "metodo": "Selenium: WebDriverWait y búsqueda dentro de article",
        "paginas": paginas, "resultados": filas,
    }
    guardar_json(FUENTES / f"busqueda_2023_{mes:02d}.json", captura)
    return captura

In [5]:
faltantes = [m for m in range(1, 6)
             if not (FUENTES / f"busqueda_2023_{m:02d}.json").exists()]
if faltantes:
    opciones = webdriver.ChromeOptions()
    opciones.add_argument("--headless=new")  # Quitar para observar Chrome.
    opciones.add_argument("--window-size=1400,1000")
    opciones.add_argument("user-agent=Mozilla/5.0")
    opciones.page_load_strategy = "eager"
    navegador = webdriver.Chrome(options=opciones)
    navegador.set_page_load_timeout(60)
    try:
        for mes in faltantes:
            extraer_mes(navegador, mes)
    finally:
        navegador.quit()
else:
    print("Se reutilizan las cinco capturas originales obtenidas con Selenium.")

capturas = [json.loads((FUENTES / f"busqueda_2023_{m:02d}.json").read_text(
    encoding="utf-8")) for m in range(1, 6)]
print("Ejemplo de una tarjeta de enero:")
display(pd.DataFrame(capturas[0]["resultados"]).head(1))

Se reutilizan las cinco capturas originales obtenidas con Selenium.
Ejemplo de una tarjeta de enero:


,numero,fecha_texto,titulo_lista,enlace
0,Decreto Supremo N.° 017-2023-PCM,31 de enero de 2023,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


## Paso 4. Verificar los cinco meses antes de filtrar
La igualdad se comprueba antes de quitar duplicados y resultados ajenos a PCM.
Después de filtrar sería normal tener menos filas que resultados del buscador.

In [6]:
verificacion = pd.DataFrame([{
    "mes": c["mes"], "resultados_totales": c["resultados_totales"],
    "resultados_extraidos": len(c["resultados"]),
} for c in capturas])
display(verificacion)
assert (verificacion["resultados_totales"] == verificacion["resultados_extraidos"]).all()
verificacion.to_csv(DATOS / "verificacion_mensual.csv", index=False, encoding="utf-8-sig")
filas = []
for captura in capturas:
    filas.extend(captura["resultados"])
brutos = pd.DataFrame(filas)
print("Total de resultados:", len(brutos))

,mes,resultados_totales,resultados_extraidos
0,2023-01,11,11
1,2023-02,9,9
2,2023-03,14,14
3,2023-04,11,11
4,2023-05,8,8


Total de resultados: 53


## Paso 5. Quitar duplicados y otras instituciones
No basta con el filtro visual PCM: se verifica el enlace. Se conserva el número tal
como aparece en la fuente, incluido `0039-2023-PCM`. Una clave numérica auxiliar
permite reconocer que `0039` y `039` designarían el mismo decreto si aparecieran ambos.

In [7]:
sin_duplicados = brutos.drop_duplicates(subset="enlace").copy()
print("Enlaces repetidos eliminados:", len(brutos) - len(sin_duplicados))
es_pcm = sin_duplicados["enlace"].str.contains("/institucion/pcm/", regex=False)
print("Normas de otras instituciones eliminadas:", int((~es_pcm).sum()))
display(sin_duplicados.loc[~es_pcm, ["numero", "enlace"]])
pcm = sin_duplicados.loc[es_pcm].copy()

def clave_norma(numero):
    coincidencia = re.search(r"(\d+)-(\d{4})-PCM", numero)
    if numero.startswith("Decreto Supremo") and coincidencia:
        return f"DS-{int(coincidencia.group(1))}-{coincidencia.group(2)}-PCM"
    return numero

pcm["clave"] = pcm["numero"].apply(clave_norma)
antes = len(pcm)
pcm = pcm.drop_duplicates("clave").copy()
print("Normas repetidas adicionales eliminadas:", antes - len(pcm))
print("Normas únicas de PCM:", len(pcm))

meses = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5}
def convertir_fecha(texto):
    dia, mes, anio = re.fullmatch(r"(\d{1,2}) de ([a-z]+) de (\d{4})", texto).groups()
    return pd.Timestamp(year=int(anio), month=meses[mes], day=int(dia))

pcm["fecha"] = pcm["fecha_texto"].apply(convertir_fecha)
assert pcm["fecha"].between("2023-01-01", "2023-05-31").all()

Enlaces repetidos eliminados: 0
Normas de otras instituciones eliminadas: 4


,numero,enlace
3,Resolución N.° 0209-2023-TCE-S1,https://www.gob.pe/institucion/oece/normas-leg...
10,Resolución N.° 0090-2023-TCE-S1,https://www.gob.pe/institucion/oece/normas-leg...
20,Resolución N.° 1665-2023-TCE-S2,https://www.gob.pe/institucion/oece/normas-leg...
39,Resolución N.° 1820-2023-TCE-S4,https://www.gob.pe/institucion/oece/normas-leg...


Normas repetidas adicionales eliminadas: 0
Normas únicas de PCM: 49


## Paso 6. Leer el título completo en cada página
Los textos de la lista se guardan como `titulo_lista`. El texto de `div.description`
se guarda como `titulo_web`. Se retira el bloque de archivos adjuntos para que sus
nombres no influyan en la clasificación. `.decompose()` es un complemento de BeautifulSoup:
elimina ese bloque del objeto HTML que estamos procesando, sin modificar el sitio.

In [8]:
def titulo_completo(enlace):
    identificador = enlace.rstrip("/").split("/")[-1]
    html = descargar_texto(enlace, FUENTES / "normas" / f"{identificador}.html")
    sopa = BeautifulSoup(html, "html.parser")
    descripcion = sopa.find("div", class_="description")
    assert descripcion is not None, f"Falta description: {enlace}"
    for bloque in descripcion.select(".institution-document__files"):
        bloque.decompose()
    return descripcion.get_text(" ", strip=True)

pcm["titulo_web"] = pcm["enlace"].apply(titulo_completo)
assert pcm["titulo_web"].str.len().gt(0).all()
display(pcm[["numero", "titulo_lista", "titulo_web"]].head(5))

,numero,titulo_lista,titulo_web
0,Decreto Supremo N.° 017-2023-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
1,Decreto Supremo N.° 016-2023-PCM,Decreto Supremo que declara el Estado de Emerg...,Decreto Supremo que declara el Estado de Emerg...
2,Decreto Supremo N.° 014-2023-PCM,Decreto Supremo que prorroga el estado de emer...,Decreto Supremo que prorroga el estado de emer...
4,Decreto Supremo N.° 010-2023-PCM,Decreto Supremo que declara el Estado de Emerg...,Decreto Supremo que declara el Estado de Emerg...
5,Directiva N.° 005-2015-PCM/SGRD,"""Directiva para la ejecución del ejercicio de ...","""Directiva para la ejecución del ejercicio de ..."


## Paso 7. Aplicar las reglas textuales solicitadas
`lower()` facilita la comparación. Se evalúa primero `prórroga`/`prorroga`, porque
una prórroga también puede mencionar que el estado de emergencia fue "declarado".
La variable `motivo` conserva `otro` cuando ninguna de las dos frases aparece.

In [9]:
def clasificar(texto):
    texto = texto.lower()
    es_lluvia = "lluvia" in texto or "precipitaciones" in texto
    if "prórroga" in texto or "prorroga" in texto:
        tipo = "prorroga"
    elif "declara" in texto:
        tipo = "declara"
    else:
        tipo = "otro"
    if "peligro inminente" in texto:
        motivo = "peligro inminente"
    elif "impacto de daños" in texto:
        motivo = "impacto de daños"
    else:
        motivo = "otro"
    return pd.Series({"es_lluvia": es_lluvia, "tipo": tipo, "motivo": motivo})

literal = pcm.copy()
literal[ ["es_lluvia", "tipo", "motivo"] ] = literal["titulo_web"].apply(clasificar)
print("Normas de lluvia según la descripción web:", int(literal["es_lluvia"].sum()))
display(literal.loc[(literal["tipo"] == "otro") | (literal["motivo"] == "otro"),
                    ["numero", "titulo_web", "es_lluvia", "tipo", "motivo"]])

Normas de lluvia según la descripción web: 20


,numero,titulo_web,es_lluvia,tipo,motivo
1,Decreto Supremo N.° 016-2023-PCM,Decreto Supremo que declara el Estado de Emerg...,False,declara,otro
4,Decreto Supremo N.° 010-2023-PCM,Decreto Supremo que declara el Estado de Emerg...,False,declara,otro
5,Directiva N.° 005-2015-PCM/SGRD,"""Directiva para la ejecución del ejercicio de ...",False,declara,otro
6,Directiva N.° 004-2015-PCM/SGRD,"""Directiva para la ejecución de simulacro noct...",False,declara,otro
7,Decreto Supremo N.° 009-2023-PCM,Decreto Supremo que declara el Estado de Emerg...,False,declara,otro
11,Decreto Supremo N.° 027-2023-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
12,Decreto Supremo N.° 025-2023-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
14,Decreto Supremo N.° 023-2023-PCM,Decreto Supremo que declara el Estado de Emerg...,False,declara,otro
15,Decreto Supremo N.° 022-2023-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
16,Decreto Supremo N.° 021-2023-PCM,Prórroga del Estado de Emergencia en distritos...,False,prorroga,otro


* Revisión de `otro`: Las directivas 004 y 005 de 2015 no son declaratorias ni prórrogas
y no se incluyen entre los decretos de lluvias. La regla parcial `"declara" in texto` las etiqueta como `declara` por la palabra "declarados". Esa etiqueta no cambia su naturaleza documental y `es_lluvia=False` las mantiene fuera del análisis. Varias normas de seguridad u orden interno
no indican los motivos meteorológicos pedidos. 

El DS 067-2023-PCM habla de déficit hídrico
y tampoco cumple la regla de lluvia. Los DS 043 y 065 de 2023 sí mencionan precipitaciones,
pero dicen "por desastre de gran magnitud". Su motivo queda en `otro` bajo la regla literal.
No se los fuerza a "impacto de daños" para que los porcentajes sumen artificialmente 100%.

* Problema detectado y complemento de verificación documental: El DS 036-2023-PCM
queda fuera con la descripción web "Declaratoria de Estado de Emergencia". La página 1
de su [PDF oficial](https://cdn.www.gob.pe/uploads/document/file/4310248/DS%20N%C2%B0%20036-2023-PCM.pdf.pdf?v=1679580651)
identifica daños por precipitaciones en Ancón, Pucusana, Punta Hermosa, Punta Negra,
San Bartolo y Santa María del Mar (Lima). Se transcribe su título y se conserva la versión
web en la auditoría. La transcripción adapta únicamente mayúsculas y puntuación para lectura.

También se inspeccionaron las primeras páginas de los PDF 046 y 055, cuyos títulos no
explican el motivo: se refieren al orden interno/seguridad y se mantienen fuera.
Esto es una revisión complementaria de casos problemáticos, no una lectura exhaustiva
de todos los artículos y anexos de todas las normas.

In [10]:
pcm["titulo"] = pcm["titulo_web"]
pcm["fuente_titulo"] = "descripcion_web"
corregir = pcm["clave"].eq("DS-36-2023-PCM")
assert corregir.sum() == 1
pcm.loc[corregir, "titulo"] = (
    "Decreto Supremo que declara el Estado de Emergencia en los distritos de Ancón, "
    "Pucusana, Punta Hermosa, Punta Negra, San Bartolo y Santa María del Mar de la "
    "provincia y departamento de Lima, por impacto de daños ante intensas precipitaciones pluviales"
)
pcm.loc[corregir, "fuente_titulo"] = "PDF oficial DS 036-2023-PCM, pagina 1"
pcm[["es_lluvia", "tipo", "motivo"]] = pcm["titulo"].apply(clasificar)
display(pcm.loc[corregir, ["numero", "titulo_web", "titulo", "fuente_titulo"]])

columnas = ["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]
solo_web = literal.loc[literal["es_lluvia"]].rename(columns={"titulo_web": "titulo"})
solo_web[columnas].to_csv(DATOS / "decretos_solo_regla_web.csv", index=False,
                         encoding="utf-8-sig", date_format="%Y-%m-%d")
lluvias = pcm.loc[pcm["es_lluvia"]].copy()
assert lluvias["numero"].str.startswith("Decreto Supremo").all()
print("Decretos después de verificar el PDF:", len(lluvias))
print(lluvias["tipo"].value_counts())
display(lluvias[columnas].head(5))
print("Motivos pendientes de una categoría literal específica:")
display(lluvias.loc[lluvias["motivo"].eq("otro"), ["numero", "titulo", "tipo", "motivo"]])

,numero,titulo_web,titulo,fuente_titulo
26,Decreto Supremo N.° 036-2023-PCM,Declaratoria de Estado de Emergencia,Decreto Supremo que declara el Estado de Emerg...,"PDF oficial DS 036-2023-PCM, pagina 1"


Decretos después de verificar el PDF: 21
tipo
declara     18
prorroga     3
Name: count, dtype: int64


,numero,fecha,titulo,tipo,motivo,enlace
8,Decreto Supremo N.° 008-2023-PCM,2023-01-13,Decreto Supremo que declara el Estado de Emerg...,declara,peligro inminente,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 007-2023-PCM,2023-01-13,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-lega...
13,Decreto Supremo N.° 024-2023-PCM,2023-02-17,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-lega...
18,Decreto Supremo N.° 019-2023-PCM,2023-02-06,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-lega...
21,Decreto Supremo N.° 044-2023-PCM,2023-03-29,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-lega...


Motivos pendientes de una categoría literal específica:


,numero,titulo,tipo,motivo
22,Decreto Supremo N.° 043-2023-PCM,Decreto Supremo que declara el Estado de Emerg...,declara,otro
46,Decreto Supremo N.° 065-2023-PCM,Decreto Supremo que prorroga el Estado de Emer...,prorroga,otro


## Paso 8. Identificar departamentos sin contar dos veces
Se recorre la lista de 25 unidades y se pregunta si cada nombre aparece **como palabra
completa** en el título original. Una coincidencia es una decisión sí/no. No se cuenta
cuántas veces aparece el nombre dentro del mismo decreto.

Por ejemplo, el DS 008-2023-PCM menciona Huancavelica, pero no Ica. La búsqueda ingenua
`"ica" in titulo.lower()` lo clasificaría mal. El DS 044-2023-PCM menciona Ica como provincia
y como departamento; nuestro procedimiento aporta una sola fila para Ica.

In [11]:
def encontrar_departamentos(titulo):
    encontrados = []
    for departamento in departamentos:
        if re.search(r"\b" + re.escape(departamento) + r"\b", titulo):
            encontrados.append(departamento)
    return encontrados

lluvias["departamentos"] = lluvias["titulo"].apply(encontrar_departamentos)
assert lluvias["departamentos"].apply(len).gt(0).all()
display(lluvias[["numero", "departamentos"]])

t8 = lluvias.loc[lluvias["clave"].eq("DS-8-2023-PCM"), "titulo"].iloc[0]
t44 = lluvias.loc[lluvias["clave"].eq("DS-44-2023-PCM"), "titulo"].iloc[0]
print("DS 008: búsqueda ingenua de ica:", "ica" in t8.lower())
print("DS 008: departamentos correctos:", encontrar_departamentos(t8))
print("DS 044: menciones textuales de Ica:", len(re.findall(r"\bIca\b", t44)))
print("DS 044: veces que contamos Ica:", encontrar_departamentos(t44).count("Ica"))
assert "Ica" not in encontrar_departamentos(t8)
assert encontrar_departamentos(t44).count("Ica") == 1

,numero,departamentos
8,Decreto Supremo N.° 008-2023-PCM,"[Amazonas, Áncash, Arequipa, Ayacucho, Cajamar..."
9,Decreto Supremo N.° 007-2023-PCM,"[Amazonas, Áncash, Cusco, Huánuco, La Libertad..."
13,Decreto Supremo N.° 024-2023-PCM,[Lima]
18,Decreto Supremo N.° 019-2023-PCM,[Arequipa]
21,Decreto Supremo N.° 044-2023-PCM,"[Huánuco, Ica]"
22,Decreto Supremo N.° 043-2023-PCM,"[Lambayeque, Piura, Tumbes]"
23,Decreto Supremo N.° 0039-2023-PCM,"[Amazonas, Ayacucho, Pasco, Ucayali]"
24,Decreto Supremo N.° 038-2023-PCM,"[Áncash, Huancavelica, Ica, Lima]"
26,Decreto Supremo N.° 036-2023-PCM,[Lima]
27,Decreto Supremo N.° 035-2023-PCM,"[Áncash, Apurímac, Arequipa, Ayacucho, Callao,..."


DS 008: búsqueda ingenua de ica: True
DS 008: departamentos correctos: ['Amazonas', 'Áncash', 'Arequipa', 'Ayacucho', 'Cajamarca', 'Huancavelica', 'Huánuco', 'Junín', 'La Libertad', 'Lima', 'Loreto', 'Moquegua', 'Pasco', 'San Martín', 'Tacna', 'Ucayali']
DS 044: menciones textuales de Ica: 2
DS 044: veces que contamos Ica: 1


Los 21 títulos incluidos usan los nombres y tildes de la lista.
La tabla anterior permite cotejarlos uno a uno. Los límites de palabra previenen la
confusión Ica/Huancavelica, pero por sí solos no resuelven todos los nombres de distritos,
provincias y departamentos que puedan coincidir: esa es una limitación del método.

## Paso 9. Contar y guardar
Una fila de la tabla larga representa un par **decreto–departamento**. Un decreto con
varios departamentos aporta uno a cada departamento, por lo que sumar los conteos
departamentales no da el número de decretos nacionales únicos.

In [12]:
filas_departamento = []
for _, fila in lluvias.iterrows():
    for departamento in fila["departamentos"]:
        filas_departamento.append({
            "numero": fila["numero"], "departamento": departamento,
            "declaratorias": int(fila["tipo"] == "declara"),
            "prorrogas": int(fila["tipo"] == "prorroga"),
        })
detalle = pd.DataFrame(filas_departamento)
assert not detalle.duplicated(["numero", "departamento"]).any()
resumen = detalle.groupby("departamento")[["declaratorias", "prorrogas"]].sum().reset_index()
resumen = pd.DataFrame({"departamento": departamentos}).merge(
    resumen, on="departamento", how="left", validate="one_to_one")
resumen[["declaratorias", "prorrogas"]] = resumen[["declaratorias", "prorrogas"]].fillna(0).astype(int)
display(resumen)
lluvias[columnas].sort_values(["fecha", "numero"]).to_csv(
    DATOS / "decretos_lluvias.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
resumen.to_csv(DATOS / "decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
pcm.drop(columns=["clave"]).to_csv(DATOS / "auditoria_normas_pcm.csv", index=False,
                                   encoding="utf-8-sig", date_format="%Y-%m-%d")
detalle.to_csv(DATOS / "detalle_decreto_departamento.csv", index=False, encoding="utf-8-sig")
print("Archivos de la Parte 1 guardados. No se cuentan las prórrogas como nuevas declaratorias.")

,departamento,declaratorias,prorrogas
0,Amazonas,3,1
1,Áncash,6,1
2,Apurímac,1,0
3,Arequipa,4,0
4,Ayacucho,5,1
5,Cajamarca,4,0
6,Callao,1,0
7,Cusco,2,0
8,Huancavelica,3,0
9,Huánuco,4,0


Archivos de la Parte 1 guardados. No se cuentan las prórrogas como nuevas declaratorias.


En esta captura se muestran 53 resultados; 0 enlaces duplicados; 4 resultados ajenos a PCM;
49 normas de PCM; 20 decretos de lluvias detectados por el título web; 21 después de
incorporar el DS 036 verificado en el PDF. La base analítica tiene 18 declaratorias y
3 prórrogas. Se mantiene también `decretos_solo_regla_web.csv` para comparar la variante
estrictamente automática (17 declaratorias y 3 prórrogas).